# Project Track 4 - Uncertainty and Data-Sufficiency Study

**Choose one variant:**

- **4A Training-seed uncertainty:** repeat the same DNN training and build an ensemble uncertainty map.
- **4B Training-case sufficiency:** hold architecture fixed and vary only the number of Reynolds-number cases.

The goal is to distinguish one lucky training run from a reproducible scientific result.

## Required files
`P4_Uncertainty_Study.ipynb`, `w4utils.py`, `w5_common.py`, `cavity_data.npz`

In [ ]:
import time,importlib
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import w4utils,w5_common
importlib.reload(w4utils);importlib.reload(w5_common)
assert w4utils.W4_UTILS_VERSION == "6.3", "Upload the revised w4utils.py (v6.3)."
assert w5_common.W5_COMMON_VERSION == "1.2", "Upload the revised w5_common.py (v1.2)."
data=w5_common.require_week4_files(); VARIANT="4A"
VAL_RE=300; TEST_RE=[175,275,375]; HIDDEN=(64,64,64)

## 4A. Repeated training seeds

Architecture, data, optimizer, and stopping rule remain fixed. Only initialization and stochastic optimization change.

In [ ]:
SEEDS=[11,22,33,44,55]
BASE_TRAIN=[100,150,200,225,250,350,400]
seed_rows=[];seed_predictions={}
if VARIANT=="4A":
    for seed in SEEDS:
        t0=time.time()
        b=w5_common.train_pointwise_model(data,BASE_TRAIN,VAL_RE,hidden=HIDDEN,
            stride=2,seed=seed,epochs=850,patience=60)
        seed_predictions[seed]={}
        for r in TEST_RE:
            pred=w5_common.predict_case(b,r,data["x"],data["y"])
            seed_predictions[seed][r]=pred
            seed_rows.append({"seed":seed,"Re":r,"training_seconds":time.time()-t0,
                              **w5_common.evaluate_prediction(data,r,pred)})
    seed_results=pd.DataFrame(seed_rows);display(seed_results)

## 4B. Number of development cases

The architecture and spatial sampling stay fixed. Only the number and coverage of physical cases change. The labels below count **development cases**, including the fixed validation case at `Re=300`; the result table reports both development-case and actual training-case counts.

In [ ]:
DEV_CASE_SETS={
    "4_dev_cases":[100,200,300,400],
    "6_dev_cases":[100,150,200,250,300,400],
    "8_dev_cases":[100,150,200,225,250,300,350,400],
}
size_rows=[];size_predictions={}
if VARIANT=="4B":
    for label,cases in DEV_CASE_SETS.items():
        val=300; train=[r for r in cases if r!=val]
        t0=time.time()
        b=w5_common.train_pointwise_model(data,train,val,hidden=HIDDEN,
            stride=2,seed=690,epochs=850,patience=60)
        size_predictions[label]={}
        for r in TEST_RE:
            pred=w5_common.predict_case(b,r,data["x"],data["y"])
            size_predictions[label][r]=pred
            size_rows.append({"case_set":label,"n_dev_cases":len(cases),
                              "n_train_cases":len(train),"Re":r,
                              "training_seconds":time.time()-t0,
                              **w5_common.evaluate_prediction(data,r,pred)})
    size_results=pd.DataFrame(size_rows);display(size_results)

## 3. Ensemble or data-sufficiency evidence

In [ ]:
if VARIANT=="4A":
    r=275; stack_u=np.stack([seed_predictions[s][r][0] for s in SEEDS]);
    stack_v=np.stack([seed_predictions[s][r][1] for s in SEEDS]);
    stack_p=np.stack([seed_predictions[s][r][2] for s in SEEDS]);
    ensemble=(stack_u.mean(0),stack_v.mean(0),stack_p.mean(0))
    idx=int(np.where(data["Re"]==r)[0][0])
    spread=np.sqrt(stack_u.std(0)**2+stack_v.std(0)**2)
    error=np.hypot(ensemble[0]-data["u"][idx],ensemble[1]-data["v"][idx])
    corr=np.corrcoef(spread.ravel(),error.ravel())[0,1]
    print("correlation between ensemble spread and actual vector error:",corr)
    X,Y=np.meshgrid(data["x"],data["y"])
    fig,ax=plt.subplots(1,3,figsize=(13,3.7))
    a=ax[0].contourf(X,Y,error,28);fig.colorbar(a,ax=ax[0]);ax[0].set_title("ensemble-mean error")
    a=ax[1].contourf(X,Y,spread,28);fig.colorbar(a,ax=ax[1]);ax[1].set_title("training-seed spread")
    ax[2].scatter(spread.ravel(),error.ravel(),s=5,alpha=.4);ax[2].set(xlabel="spread",ylabel="error",title=f"correlation={corr:.2f}")
    plt.tight_layout();plt.show()
    seed_results.to_csv("P4_seed_results.csv",index=False)
else:
    size_results.to_csv("P4_data_size_results.csv",index=False)
    fig,ax=plt.subplots(1,2,figsize=(10,3.8))
    for label,g in size_results.groupby("case_set"):
        ax[0].plot(g["Re"],g["relative_L2_uv"],"o-",label=label)
        ax[1].plot(g["Re"],g["relative_L2_p"],"o-",label=label)
    for a in ax:a.grid(.3);a.legend();a.set_xlabel("blind Re")
    ax[0].set_ylabel("relative L2 velocity");ax[1].set_ylabel("relative L2 pressure")
    plt.tight_layout();plt.show()

## Required report evidence

### Variant 4A
- Report mean, standard deviation, best, and worst error across seeds.
- Compare one model with the ensemble mean.
- Plot uncertainty/spread and actual error on the same blind case.
- Explain whether ensemble spread is a useful error indicator and where it fails.

### Variant 4B
- Keep network and training rules fixed.
- Report error versus both the number of development cases and the actual number of training cases, not merely the number of flattened grid points.
- Explain whether added cases improve interpolation, extrapolation, or both.
- Identify the smallest case set that preserves the central flow but loses a wall/corner feature.

For both variants, a result without repeated runs or a controlled data change is not an uncertainty study.

## Optional stretch extensions (advanced / prize-track only)

Complete the required project first. With instructor approval, choose at most one:

1. Evaluate whether ensemble spread is calibrated by plotting empirical error coverage at several spread thresholds.
2. Combine training-seed variability with development-case count in a small two-factor experiment.
3. Use ensemble disagreement as an acquisition score and test one simple active-learning choice against a random added Reynolds case.
